# MMS_3D — method-of-manufactured-solutions forms, merged (3D twins)

`MagneticStandalone3D` · ASN 28 validation form (TRUST-LiMe).

Merged form (notebook-merging pass M4, second half): the four historical
3D MMS notebooks — `MMS_Diff3D`, `MMS_Trig3D`, `MMS_ExpTrig3D`,
`MMS_Poly_3D` — run as four datasets of a single form folder, each
keeping its own baseline checks (copied verbatim) and its
mesh-convergence scan (per-case `build/<case>/<scale>/`
subdirectories). The datasets are byte-identical to the historical ones.

In [ ]:
from trustutils import run
import numpy as np
import os
from trustutils.files import SonFile

run.TRUST_parameters("1.9.8")
run.reset()

NB_PROCS = 2
BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)
print("BUILD =", BUILD)

In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder, per-case subdirectories) ---------
# Each single dataset src/<Case>.data carries $nx/$ny/$nz node-count
# placeholders; every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<Case>/<scale>/ —
# one data-file name across scales:
#   coarse   = the baseline mesh
#   standard = the refined mesh (the historical L1 level)
#   fine     = the finest mesh (the historical L2 level)
cases = {}
for base, lv in (
    ("MMS_Diff3D", {
        "coarse": {"nx": 21, "ny": 21, "nz": 21},
        "standard": {"nx": 21, "ny": 41, "nz": 21},
        "fine": {"nx": 21, "ny": 81, "nz": 21},
    }),
    ("MMS_Trig3D", {
        "coarse": {"nx": 5, "ny": 5, "nz": 5},
        "standard": {"nx": 9, "ny": 9, "nz": 9},
        "fine": {"nx": 17, "ny": 17, "nz": 17},
    }),
    ("MMS_ExpTrig3D", {
        "coarse": {"nx": 5, "ny": 5, "nz": 5},
        "standard": {"nx": 9, "ny": 9, "nz": 5},
        "fine": {"nx": 17, "ny": 17, "nz": 5},
    }),
    ("MMS_Poly_3D", {
        "coarse": {"nx": 5, "ny": 5, "nz": 5},
        "standard": {"nx": 9, "ny": 9, "nz": 9},
        "fine": {"nx": 17, "ny": 17, "nz": 17},
    }),
):
    cases[base] = {}
    for lvl, dic in lv.items():
        cases[base][lvl] = run.addCaseFromTemplate(
            f"{base}.data", f"{base}/{lvl}", dic=dic,
            nbProcs=NB_PROCS, excluNR=True)
        cases[base][lvl].partition()   # trust -partition <Case>.data 2 -> PAR_<Case>.data + Zones
run.printCases()
run.runCases()
c_diff3d = cases["MMS_Diff3D"]["coarse"]
c_trig3d = cases["MMS_Trig3D"]["coarse"]
c_exptrig3d = cases["MMS_ExpTrig3D"]["coarse"]
c_poly3d = cases["MMS_Poly_3D"]["coarse"]


## MMS_Diff3D — baseline checks (verbatim from the historical form)

In [ ]:
import os
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# MMS_Diff3D: b(x, y, 0) = sin(pi y) with Dirichlet zero at y = 0, 1 and
# Neumann walls elsewhere is an exact eigenmode of the diffusion operator:
#     b(x, y, t) = sin(pi y) exp(-eta pi^2 t),  eta = 0.05.
# All three probes sit on the mode, so each must follow the same analytic
# decay.  The expected ratio uses the ACTUAL final time of the run.
eta = 0.05

for name in ('SONDE_Y025', 'SONDE_Y050', 'SONDE_Y075'):
    t, b = read_son(c_diff3d.dataFileName_, name)
    ratio = b[-1] / b[0]
    expected = np.exp(-eta * np.pi**2 * t[-1])
    err = abs(ratio - expected) / expected
    print(f"{name}: ratio={ratio:.4f} expected={expected:.4f} err={err:.3e}")
    assert err < 0.10, f"{name} decay error too large: {err:.3e}"
print("OK: sin(pi y) mode decays at the analytic eta pi^2 rate at all probes.")

## Mesh convergence

The shared refinement ladder, adapted to the 1D eigenmode physics: only the
active `y` direction is refined (`coarse` = the baseline mesh `21×21×21` — the
dataset already sits at 21³ because coarser meshes excite higher discrete
P0 modes — `standard` = y cells ×2 (`21×41×21`), `fine` = y cells ×4 (`21×81×21`)).

The *Checks* ratio metric saturates at the implicit-Euler temporal error,
common to every mesh. To expose the **spatial** error, the scan compares
$b(t)/b(0)$ against the exact implicit-Euler decay built with the analytic
eigenvalue $\lambda = \eta\pi^2$ on the probe's own time grid: the time
scheme cancels out and the residual is the spatial one,
$\lambda_h - \lambda = O(h^2)$. The observed order is $\log_2(e_k/e_{k+1})$.

In [ ]:
# merged-form aliases (the historical scan cells, namespaced)
scan_cases = cases["MMS_Diff3D"]
SONDE_TAGS_SCAN = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
ETA = 0.05                       # magnetic diffusivity of the dataset

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def implicit_ref(t, lam):
    """Exact implicit-Euler decay ratio on the probe time grid (any dt sequence)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 / (1.0 + lam * (t[i] - t[i-1]))
    return ref

LAM = ETA * np.pi**2                # analytic eigenvalue eta*pi^2
def spatial_ratio_error(subdir, case_name):
    """Max |b(t)/b(0) - implicit-Euler analytic decay| over the scan probes."""
    worst = 0.0
    for tag in SONDE_TAGS_SCAN:
        t, b = read_son_dir(subdir, case_name, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - implicit_ref(t, LAM)))))
    return worst

scan = {"coarse": ("MMS_Diff3D/coarse", c_diff3d.dataFileName_),
        "standard": (f"MMS_Diff3D/standard", scan_cases["standard"].dataFileName_),
        "fine": (f"MMS_Diff3D/fine", scan_cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (21, 21, 21), "standard": (21, 41, 21), "fine": (21, 81, 21)}
scan_err = {lvl: spatial_ratio_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes       h_y        |ratio - implicit ref|   observed order")
prev = None
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    e = scan_err[lvl]
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {1.0/(ny-1):<10.4g} {e:<26.4e} {order}")
    prev = e

assert scan_err["standard"] < scan_err["coarse"], f"no convergence coarse -> standard: {scan_err}"
assert scan_err["fine"] < scan_err["standard"], f"no convergence standard -> fine: {scan_err}"
print("OK: spatial error decreases monotonically under refinement (order ~2).")

## MMS_Trig3D — baseline checks (verbatim from the historical form)

In [ ]:
import os
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# The dataset as written is an invariance case: the uniform initial field
# b0 = (0, 1) satisfies u.grad(b) = 0 (u = 0 after the component-count fix),
# its Laplacian vanishes, and the walls hold (0, 1).  Both components must
# stay at their initial values to machine precision.
tbx, bx = read_son(c_trig3d.dataFileName_, 'SONDE_BX')
tby, by = read_son(c_trig3d.dataFileName_, 'SONDE_BY')

err_bx = np.max(np.abs(bx))
err_by = np.max(np.abs(by - 1.0))
print(f"max |Bx|          : {err_bx:.3e}")
print(f"max |By - 1|      : {err_by:.3e}")
print(f"final time        : {tbx[-1]:.6g}")
assert err_bx < 1e-10, f"Bx not preserved: {err_bx:.3e}"
assert err_by < 1e-10, f"By not preserved: {err_by:.3e}"
print("OK: uniform (0, 1) field is invariant under convection + diffusion.")

## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`5×5×5`),
`standard` = cells ×2 (`9×9×9`), `fine` = cells ×4 (`17×17×17`).

The dataset is an **invariance case** (uniform (0, 1) field, convection on): the scan asserts the machine-precision identity at every level, no order reported.

In [ ]:
# merged-form aliases (the historical scan cells, namespaced)
scan_cases = cases["MMS_Trig3D"]


def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def level_error(subdir, case_name):
    """(max |Bx|, max |By - 1|) at the scan probes."""
    _, bx = read_son_dir(subdir, case_name, "SONDE_BX")
    _, by = read_son_dir(subdir, case_name, "SONDE_BY")
    return float(np.max(np.abs(bx))), float(np.max(np.abs(by - 1.0)))

scan = {"coarse": ("MMS_Trig3D/coarse", c_trig3d.dataFileName_),
        "standard": (f"MMS_Trig3D/standard", scan_cases["standard"].dataFileName_),
        "fine": (f"MMS_Trig3D/fine", scan_cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (5, 5, 5), "standard": (9, 9, 9), "fine": (17, 17, 17)}
scan_err = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes      max |Bx|        max |By - 1|")
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    ex, ey = scan_err[lvl]
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {ex:<14.3e} {ey:.3e}")
    assert ex < 1e-10, f"Bx not preserved at {lvl}: {ex:.3e}"
    assert ey < 1e-10, f"By not preserved at {lvl}: {ey:.3e}"
print("OK: uniform (0, 1) field invariance holds at every mesh level.")

## MMS_ExpTrig3D — baseline checks (verbatim from the historical form)

In [ ]:
import os
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# MMS_ExpTrig3D: b(x, y, 0) = sin(pi x) exp(-y), u = 0, eta = 0.01, with
# Neumann walls in x/z and thin Dirichlet layers at y = 0, 1.  Away from the
# walls the local decay rate is eta * |lap b| / b = eta * (pi^2 - 1); over the
# nb_pas_dt_max-capped run the probe ratios must follow it to a few percent.
eta = 0.01
rate = eta * (np.pi**2 - 1.0)

for name in ('SONDE_Y025', 'SONDE_Y050', 'SONDE_Y075'):
    t, b = read_son(c_exptrig3d.dataFileName_, name)
    ratio = b[-1] / b[0]
    expected = np.exp(-rate * t[-1])
    err = abs(ratio - expected) / expected
    print(f"{name}: ratio={ratio:.4f} expected={expected:.4f} err={err:.3e}")
    assert err < 0.10, f"{name} decay error too large: {err:.3e}"
print("OK: exp-trig decay matches the local-rate prediction at all probes.")

## Mesh convergence

The shared refinement ladder, adapted to the field's plane: $b_0 = \sin(\pi x)e^{-y}$
varies in `x` and `y` only, so `z` keeps its baseline node count (`coarse` =
`5×5×5`, `standard` = x,y cells ×2 (`9×9×5`), `fine` = x,y cells ×4 (`17×17×5`)).

The *Checks* ratio metric saturates at the implicit-Euler temporal error,
common to every mesh. To expose the **spatial** error, the scan compares
$b(t)/b(0)$ against the exact implicit-Euler decay built with the analytic
local rate $\lambda = \eta(\pi^2 - 1)$ on the probe's own time grid. The
observed order is $\log_2(e_k/e_{k+1})$.

In [ ]:
# merged-form aliases (the historical scan cells, namespaced)
scan_cases = cases["MMS_ExpTrig3D"]
SONDE_TAGS_SCAN = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
ETA = 0.01
LAM = ETA * (np.pi**2 - 1.0)   # analytic local decay rate eta*(pi^2 - 1)

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def implicit_ref(t, lam):
    """Exact implicit-Euler decay ratio on the probe time grid (any dt sequence)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 / (1.0 + lam * (t[i] - t[i-1]))
    return ref

def spatial_ratio_error(subdir, case_name):
    """Max |b(t)/b(0) - implicit-Euler analytic decay| over the scan probes."""
    worst = 0.0
    for tag in SONDE_TAGS_SCAN:
        t, b = read_son_dir(subdir, case_name, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - implicit_ref(t, LAM)))))
    return worst

scan = {"coarse": ("MMS_ExpTrig3D/coarse", c_exptrig3d.dataFileName_),
        "standard": (f"MMS_ExpTrig3D/standard", scan_cases["standard"].dataFileName_),
        "fine": (f"MMS_ExpTrig3D/fine", scan_cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (5, 5, 5), "standard": (9, 9, 5), "fine": (17, 17, 5)}
scan_err = {lvl: spatial_ratio_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes    |ratio - implicit ref|   observed order")
prev = None
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    e = scan_err[lvl]
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {e:<26.4e} {order}")
    prev = e

assert scan_err["standard"] < scan_err["coarse"], f"no convergence coarse -> standard: {scan_err}"
assert scan_err["fine"] < scan_err["standard"], f"no convergence standard -> fine: {scan_err}"
print("OK: spatial error decreases monotonically under refinement.")

## MMS_Poly_3D — baseline checks (verbatim from the historical form)

In [ ]:
import os
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# The dataset as written is an invariance case: the uniform initial field
# b0 = (0, 1) satisfies u.grad(b) = 0 (u = 0 after the component-count fix),
# its Laplacian vanishes, and the walls hold (0, 1).  Both components must
# stay at their initial values to machine precision.
tbx, bx = read_son(c_poly3d.dataFileName_, 'SONDE_BX')
tby, by = read_son(c_poly3d.dataFileName_, 'SONDE_BY')

err_bx = np.max(np.abs(bx))
err_by = np.max(np.abs(by - 1.0))
print(f"max |Bx|          : {err_bx:.3e}")
print(f"max |By - 1|      : {err_by:.3e}")
print(f"final time        : {tbx[-1]:.6g}")
assert err_bx < 1e-10, f"Bx not preserved: {err_bx:.3e}"
assert err_by < 1e-10, f"By not preserved: {err_by:.3e}"
print("OK: uniform (0, 1) field is invariant under convection + diffusion.")

## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`5×5×5`),
`standard` = cells ×2 (`9×9×9`), `fine` = cells ×4 (`17×17×17`).

The 3D twin initialises the **uniform (0, 1) field** (its 2D twin uses zero — they differ by design): conservation of that uniform field is a **discrete identity**, asserted at every level (the 3D dataset carries real probes, so all levels are instrumented); no order is reported.

In [ ]:
# merged-form aliases (the historical scan cells, namespaced)
scan_cases = cases["MMS_Poly_3D"]


def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def level_error(subdir, case_name):
    """(max |Bx|, max |By - 1|) at the scan probes (uniform (0,1) field)."""
    _, bx = read_son_dir(subdir, case_name, "SONDE_BX")
    _, by = read_son_dir(subdir, case_name, "SONDE_BY")
    return float(np.max(np.abs(bx))), float(np.max(np.abs(by - 1.0)))

scan = {"coarse": ("MMS_Poly_3D/coarse", c_poly3d.dataFileName_),
        "standard": (f"MMS_Poly_3D/standard", scan_cases["standard"].dataFileName_),
        "fine": (f"MMS_Poly_3D/fine", scan_cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (5, 5, 5), "standard": (9, 9, 9), "fine": (17, 17, 17)}
scan_err = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes      max |Bx|        max |By - 1|")
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    ex, ey = scan_err[lvl]
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {ex:<14.3e} {ey:.3e}")
    assert ex < 1e-10, f"Bx not preserved at {lvl}: {ex:.3e}"
    assert ey < 1e-10, f"By not preserved at {lvl}: {ey:.3e}"
print("OK: uniform (0, 1) field invariance holds at every mesh level.")

## Computer Performance

In [ ]:
run.tablePerf()